In [1]:
import os
import json
import shutil
import pandas as pd
from pprint import pprint


In [2]:
language = "es"
situation_number = 1


In [3]:
class SituationProcessor:
	def __init__(self, language: str, base_dir: str = "."):
		self.language = language

		self.base_dir = base_dir
		self.localization_dir = os.path.join(base_dir, "localization")

		self.raw_dir = os.path.join(self.localization_dir, "raw", language)
		self.structure_dir = os.path.join(self.localization_dir, "structure")

		self.data_dir = os.path.join(base_dir, "data")
		self.situations_dir = os.path.join(self.data_dir, "processed")

		self.processed_dir = os.path.join(self.localization_dir, "processed", language)

		self.metadata_path = os.path.join(base_dir, "metadata.json")

		with open(self.metadata_path, "r", encoding="utf-8") as f:
			self.metadata = json.load(f)

		self.state = {}

	def _load_situation_config(self, situation_number: int):
		config = self.metadata.get(str(situation_number))
		if not config:
			raise ValueError(f"Situation {situation_number} not found in metadata.")
		return config

	def _load_df(self, situation_number: int):
		path = os.path.join(
			self.situations_dir,
			f"situation_{situation_number}.csv"
		)
		return pd.read_csv(path)

	def process_situation(self, situation_number: int):
		config = self._load_situation_config(situation_number)

		keys = config["object"].split(".")
		file = config["file"]
		summary = config["summary"]

		df = self._load_df(situation_number)

		structure_path = os.path.join(self.structure_dir, file)

		with open(structure_path, "r", encoding="utf-8") as f:
			structure = json.load(f)

		language_path = os.path.join(self.raw_dir, file)
		with open(language_path, "r", encoding="utf-8") as f:
			data = json.load(f)
		self.state[file] = data

		obj = data
		struct = structure

		for key in keys:
			if key not in obj:
				obj[key] = {}
				
			obj = obj[key]
			struct = struct[key]

		print(obj)

		responses = []
		for choice_info in struct["choices"]:
			choice_name = choice_info["next"]

			rows = df[df["choice"] == choice_name]

			texts = []
			for _, row in rows.iterrows():
				if row["text_clean"]:
					texts.append(row["text_clean"])

				if row["was_flipped"] and row["text_flipped"]:
					texts.append(row["text_flipped"])

			responses.append({"text": texts})

		obj["summary"] = summary
		obj["responses"] = responses

		return data

	def process_multiple(self, situation_numbers: list[int]):
		results = {}

		for n in situation_numbers:
			print(f"Processing situation {n}...")
			results[n] = self.process_situation(n)

		return results

	def save(self):
		os.makedirs(self.processed_dir, exist_ok=True)

		for file, data in self.state.items():
			out_path = os.path.join(self.processed_dir, file)

			os.makedirs(os.path.dirname(out_path), exist_ok=True)

			with open(out_path, "w", encoding="utf-8") as f:
				json.dump(data, f, ensure_ascii=False, indent=4)

			print(f"Saved: {out_path}")

	def build_final(self, final_dir_name: str = "final"):
		final_dir = os.path.join(
			self.localization_dir,
			final_dir_name,
			self.language
		)

		if os.path.exists(final_dir):
			shutil.rmtree(final_dir)

		shutil.copytree(self.raw_dir, final_dir)

		for file, data in self.state.items():
			out_path = os.path.join(final_dir, file)

			os.makedirs(os.path.dirname(out_path), exist_ok=True)

			with open(out_path, "w", encoding="utf-8") as f:
				json.dump(data, f, ensure_ascii=False, indent=4)

			print(f"Updated: {out_path}")


In [4]:
processor = SituationProcessor("es")

data = processor.process_situation(1)
# pprint(data)
processor.save()
processor.build_final()


{'summary': 'Laura me ha saludado.', 'responses': [{'text': ['Igualmente, <player, encantado, encantada> de conocerte *sonríes*.', 'El gusto es mío, <player, amigo, amiga> *asientes con la cabeza*.', 'Un placer conocerte también, <player, encantado, encantada> *sonríes levemente*.']}, {'text': ['Gracias, si necesito algo ya te iré diciendo.', 'De acuerdo, ya te diré si necesito algo.', 'Perfecto, muchas gracias por avisar.']}, {'text': ['... Ah, sí, hola.', 'Eh... claro, sí.', 'Ah, hola.']}]}
Saved: .\localization\processed\es\scene1/scene1Classroom.json
Updated: .\localization\final\es\scene1/scene1Classroom.json
